In [ ]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
job_array=False;job_adjust=0
ocean_fraction=2/8
job_array=False

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [ ]:
times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
job_array=False;index_adjust=0
ocean_fraction=2/8

In [ ]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [57]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 16667, end_job = 33334


In [58]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [59]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'


var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory(globals())

Top 10 objects with highest memory usage
{'A_g': '17.73 MB', 'A_c': '17.73 MB', 'Z': '17.73 MB', 'Y': '17.73 MB', 'X': '17.73 MB', 'W': '8.87 MB', 'QCQI': '8.87 MB', 'parcel_z': '8.87 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB'}

0.11526 GB in use overall


In [60]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'ED_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['E_G','E_C','D_C','D_G']
data_dict = make_data_dict(var_names,read_type)
E_G, E_C, D_C, D_G = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory(globals())

Top 10 objects with highest memory usage
{'A_g': '17.73 MB', 'A_c': '17.73 MB', 'Z': '17.73 MB', 'Y': '17.73 MB', 'X': '17.73 MB', 'W': '8.87 MB', 'QCQI': '8.87 MB', 'parcel_z': '8.87 MB', 'E_G': '8.87 MB', 'E_C': '8.87 MB'}

0.133 GB in use overall


In [61]:
#TRACKED TRAJECTORY ENTRAINMENT/DETRAINMENT
################################################################

In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

#PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(ALL_out_arr)} CL parcels and {len(ALL_save_arr)} nonCL parcels')
print(f'SHALLOW: {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')
print(f'DEEP: {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(ALL_SBZ_out_arr)} SBZ parcels and {len(ALL_nonSBZ_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SHALLOW_SBZ_out_arr)} SBZ parcels and {len(SHALLOW_nonSBZ_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(DEEP_SBZ_out_arr)} SBZ parcels and {len(DEEP_nonSBZ_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ALL_ColdPool_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(SHALLOW_ColdPool_out_arr)} ColdPool parcels')
print(f'DEEP: {len(DEEP_ColdPool_out_arr)} ColdPool parcels')

#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'ALL_out_arr', 'ALL_save_arr',
        'SHALLOW_out_arr', 'SHALLOW_save_arr',
        'DEEP_out_arr', 'DEEP_save_arr',
        'ALL_SBZ_out_arr', 'ALL_nonSBZ_out_arr',
        'SHALLOW_SBZ_out_arr', 'SHALLOW_nonSBZ_out_arr',
        'DEEP_SBZ_out_arr', 'DEEP_nonSBZ_out_arr',
        'ALL_ColdPool_out_arr', 'SHALLOW_ColdPool_out_arr', 'DEEP_ColdPool_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [15]:
##########################################################################################
#(ALL, SHALLOW, DEEP) CL vs NonCL Tracked Entrainment Profiles

In [16]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def CLTrackedEDProfile(type,type2):

    if type2=='general':
        profile_array_e=E_G.copy() #PRECALCULATED
        profile_array_d=D_G.copy() #PRECALCULATED
    elif type2=='cloudy':
        profile_array_e=E_C.copy() #PRECALCULATED
        profile_array_d=D_C.copy() #PRECALCULATED

    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array
    
    
    zhs=data['zh'].values
    e_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    d_profile=e_profile.copy()
    net_profile=e_profile.copy()
    e_profile[:,2]=zhs;d_profile[:,2]=zhs;net_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_e=profile_array_e[ts,p-index_adjust] #JOBARRAY
        np.add.at(e_profile[:, 0], zs, vars_e)
        np.add.at(e_profile[:, 1], zs, 1)

        vars_d=profile_array_d[ts,p-index_adjust] #JOBARRAY
        np.add.at(d_profile[:, 0], zs, vars_d)
        np.add.at(d_profile[:, 1], zs, 1)

        vars_net=vars_e-vars_d
        np.add.at(net_profile[:, 0], zs, vars_net)
        np.add.at(net_profile[:, 1], zs, 1)
    
    print('done\n')
    return e_profile,d_profile,net_profile

In [17]:
for type2 in ['general', 'cloudy']:
    [CL_ALL_e,CL_ALL_d,CL_ALL_net] = CLTrackedEDProfile(type='all',type2=type2)
    [CL_SHALLOW_e,CL_SHALLOW_d,CL_SHALLOW_net] = CLTrackedEDProfile(type='shallow',type2=type2)
    [CL_DEEP_e,CL_DEEP_d,CL_DEEP_net] = CLTrackedEDProfile(type='deep',type2=type2)
    
    #SAVING
    import h5py
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'CL_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('CL_ALL_e', data=CL_ALL_e)
        h5f.create_dataset('CL_ALL_d', data=CL_ALL_d)
        h5f.create_dataset('CL_ALL_net', data=CL_ALL_net)
        h5f.create_dataset('CL_SHALLOW_e', data=CL_SHALLOW_e)
        h5f.create_dataset('CL_SHALLOW_d', data=CL_SHALLOW_d)
        h5f.create_dataset('CL_SHALLOW_net', data=CL_SHALLOW_net)
        h5f.create_dataset('CL_DEEP_e', data=CL_DEEP_e)
        h5f.create_dataset('CL_DEEP_d', data=CL_DEEP_d)
        h5f.create_dataset('CL_DEEP_net', data=CL_DEEP_net)

done

done

done

done

done

done



In [18]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def nonCLTrackedEDProfile(type,type2):

    if type2=='general':
        profile_array_e=E_G.copy() #PRECALCULATED
        profile_array_d=D_G.copy() #PRECALCULATED
    elif type2=='cloudy':
        profile_array_e=E_C.copy() #PRECALCULATED
        profile_array_d=D_C.copy() #PRECALCULATED

    if type=='all':
        out_arr=ALL_save_arr
        # after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr
        # after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr
        # after_array=DEEP_save_after_array
    
    
    zhs=data['zh'].values
    e_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    d_profile=e_profile.copy()
    net_profile=e_profile.copy()
    e_profile[:,2]=zhs;d_profile[:,2]=zhs;net_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_e=profile_array_e[ts,p-index_adjust] #JOBARRAY
        np.add.at(e_profile[:, 0], zs, vars_e)
        np.add.at(e_profile[:, 1], zs, 1)

        vars_d=profile_array_d[ts,p-index_adjust] #JOBARRAY
        np.add.at(d_profile[:, 0], zs, vars_d)
        np.add.at(d_profile[:, 1], zs, 1)

        vars_net=vars_e-vars_d
        np.add.at(net_profile[:, 0], zs, vars_net)
        np.add.at(net_profile[:, 1], zs, 1)
    
    print('done\n')
    return e_profile,d_profile,net_profile

In [19]:
for type2 in ['general', 'cloudy']:
    [nonCL_ALL_e,nonCL_ALL_d,nonCL_ALL_net] = nonCLTrackedEDProfile(type='all',type2=type2)
    [nonCL_SHALLOW_e,nonCL_SHALLOW_d,nonCL_SHALLOW_net] = nonCLTrackedEDProfile(type='shallow',type2=type2)
    [nonCL_DEEP_e,nonCL_DEEP_d,nonCL_DEEP_net] = nonCLTrackedEDProfile(type='deep',type2=type2)
    
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f'nonCL_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('nonCL_ALL_e', data=nonCL_ALL_e)
        h5f.create_dataset('nonCL_ALL_d', data=nonCL_ALL_d)
        h5f.create_dataset('nonCL_ALL_net', data=nonCL_ALL_net)
        h5f.create_dataset('nonCL_SHALLOW_e', data=nonCL_SHALLOW_e)
        h5f.create_dataset('nonCL_SHALLOW_d', data=nonCL_SHALLOW_d)
        h5f.create_dataset('nonCL_SHALLOW_net', data=nonCL_SHALLOW_net)
        h5f.create_dataset('nonCL_DEEP_e', data=nonCL_DEEP_e)
        h5f.create_dataset('nonCL_DEEP_d', data=nonCL_DEEP_d)
        h5f.create_dataset('nonCL_DEEP_net', data=nonCL_DEEP_net)

done

done

done

done

done

done



In [58]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [59]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file += f'CL_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_' + f'CL_tracked_profiles_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [60]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file += f'nonCL_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_' + f'nonCL_tracked_profiles_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [29]:
##########################################################################################
#SBZ vs nonSBZ Tracked Entrainment Profiles

In [ ]:
#FINDING MEAN CLOUD BASE #*****************************
zh=data['zh'].values
w_thresh2=0.5
qcqi_thresh=1e-6
type='all'

if type=='all':
    out_arr=ALL_out_arr.copy()
elif type=='deep':
    out_arr=DEEP_out_arr.copy()
elif type=='shallow':
    out_arr=SHALLOW_out_arr.copy()

zhs=data['zh'].values
profile_array =np.zeros((len(zhs), 2)) #column 1: var, column 2: counter, column 3: list of zhs
profile_array[:,1]=zhs;

# cloudbase_lst=[]
after=20*int(minutes) #20 minutes
for row in range(out_arr.shape[0]):
    if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
    p=out_arr[row,0]
    
    # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
    ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
    ts = np.arange(out_arr[row, 1], ts_end)
    
    zs=Z[ts,p-index_adjust]
    ys=Y[ts,p-index_adjust]
    xs=X[ts,p-index_adjust]

    ws=W[ts,p-index_adjust]
    qcqis=QCQI[ts,p-index_adjust]
    where=np.where((ws>=w_thresh2) & (qcqis>=qcqi_thresh))
    profile_array[zs[where],0]+=1
del after
# print(np.mean(cloudbase_lst))
# print(np.min(cloudbase_lst))
# plt.hist(cloudbase_lst,bins=40,orientation='horizontal');
all_cloudbase=zh[np.where(profile_array[:,0]!=0)[0][0]]
print(all_cloudbase)

In [72]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def SBZTrackedEDProfile(type,type2):

    if type2=='general':
        profile_array_e=E_G.copy() #PRECALCULATED
        profile_array_d=D_G.copy() #PRECALCULATED
    elif type2=='cloudy':
        profile_array_e=E_C.copy() #PRECALCULATED
        profile_array_d=D_C.copy() #PRECALCULATED

    if type=='all':
        out_arr=ALL_SBZ_out_arr
        # after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr
        # after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr
        # after_array=DEEP_SBZ_out_after_array
    
    
    zhs=data['zh'].values
    e_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    d_profile=e_profile.copy()
    net_profile=e_profile.copy()
    e_profile[:,2]=zhs;d_profile[:,2]=zhs;net_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_e=profile_array_e[ts,p-index_adjust] #JOBARRAY
        np.add.at(e_profile[:, 0], zs, vars_e)
        np.add.at(e_profile[:, 1], zs, 1)

        vars_d=profile_array_d[ts,p-index_adjust] #JOBARRAY
        np.add.at(d_profile[:, 0], zs, vars_d)
        np.add.at(d_profile[:, 1], zs, 1)

        vars_net=vars_e-vars_d
        np.add.at(net_profile[:, 0], zs, vars_net)
        np.add.at(net_profile[:, 1], zs, 1)
    
    print('done\n')
    return e_profile,d_profile,net_profile

In [73]:
for type2 in ['general','cloudy']:
    [SBZ_ALL_e,SBZ_ALL_d,SBZ_ALL_net] = SBZTrackedEDProfile(type='all',type2=type2)
    [SBZ_SHALLOW_e,SBZ_SHALLOW_d,SBZ_SHALLOW_net] = SBZTrackedEDProfile(type='shallow',type2=type2)
    [SBZ_DEEP_e,SBZ_DEEP_d,SBZ_DEEP_net] = SBZTrackedEDProfile(type='deep',type2=type2)
    
    # Define the file path
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f'SBZ_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('SBZ_ALL_e', data=SBZ_ALL_e)
        h5f.create_dataset('SBZ_ALL_d', data=SBZ_ALL_d)
        h5f.create_dataset('SBZ_ALL_net', data=SBZ_ALL_net)
        h5f.create_dataset('SBZ_SHALLOW_e', data=SBZ_SHALLOW_e)
        h5f.create_dataset('SBZ_SHALLOW_d', data=SBZ_SHALLOW_d)
        h5f.create_dataset('SBZ_SHALLOW_net', data=SBZ_SHALLOW_net)
        h5f.create_dataset('SBZ_DEEP_e', data=SBZ_DEEP_e)
        h5f.create_dataset('SBZ_DEEP_d', data=SBZ_DEEP_d)
        h5f.create_dataset('SBZ_DEEP_net', data=SBZ_DEEP_net)


done

done

done

done

done

done



In [74]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def nonSBZTrackedEDProfile(type,type2):

    if type2=='general':
        profile_array_e=E_G.copy() #PRECALCULATED
        profile_array_d=D_G.copy() #PRECALCULATED
    elif type2=='cloudy':
        profile_array_e=E_C.copy() #PRECALCULATED
        profile_array_d=D_C.copy() #PRECALCULATED

    if type=='all':
        out_arr=ALL_nonSBZ_out_arr
        # after_array=ALL_nonSBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr
        # after_array=SHALLOW_nonSBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr
        # after_array=DEEP_nonSBZ_out_after_array
    
    
    zhs=data['zh'].values
    e_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    d_profile=e_profile.copy()
    net_profile=e_profile.copy()
    e_profile[:,2]=zhs;d_profile[:,2]=zhs;net_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_e=profile_array_e[ts,p-index_adjust] #JOBARRAY
        np.add.at(e_profile[:, 0], zs, vars_e)
        np.add.at(e_profile[:, 1], zs, 1)

        vars_d=profile_array_d[ts,p-index_adjust] #JOBARRAY
        np.add.at(d_profile[:, 0], zs, vars_d)
        np.add.at(d_profile[:, 1], zs, 1)

        vars_net=vars_e-vars_d
        np.add.at(net_profile[:, 0], zs, vars_net)
        np.add.at(net_profile[:, 1], zs, 1)
    
    print('done\n')
    return e_profile,d_profile,net_profile

In [75]:
for type2 in ['general','cloudy']:
    [nonSBZ_ALL_e,nonSBZ_ALL_d,nonSBZ_ALL_net] = nonSBZTrackedEDProfile(type='all',type2=type2)
    [nonSBZ_SHALLOW_e,nonSBZ_SHALLOW_d,nonSBZ_SHALLOW_net] = nonSBZTrackedEDProfile(type='shallow',type2=type2)
    [nonSBZ_DEEP_e,nonSBZ_DEEP_d,nonSBZ_DEEP_net] = nonSBZTrackedEDProfile(type='deep',type2=type2)
    
    
    # Define the file path
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f'nonSBZ_tracked_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('nonSBZ_ALL_e', data=nonSBZ_ALL_e)
        h5f.create_dataset('nonSBZ_ALL_d', data=nonSBZ_ALL_d)
        h5f.create_dataset('nonSBZ_ALL_net', data=nonSBZ_ALL_net)
        h5f.create_dataset('nonSBZ_SHALLOW_e', data=nonSBZ_SHALLOW_e)
        h5f.create_dataset('nonSBZ_SHALLOW_d', data=nonSBZ_SHALLOW_d)
        h5f.create_dataset('nonSBZ_SHALLOW_net', data=nonSBZ_SHALLOW_net)
        h5f.create_dataset('nonSBZ_DEEP_e', data=nonSBZ_DEEP_e)
        h5f.create_dataset('nonSBZ_DEEP_d', data=nonSBZ_DEEP_d)
        h5f.create_dataset('nonSBZ_DEEP_net', data=nonSBZ_DEEP_net)

done

done

done

done

done

done



In [80]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [81]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+= f'SBZ_tracked_profiles_{t_res}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_' + f'SBZ_tracked_profiles_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [82]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+= f'nonSBZ_tracked_profiles_{t_res}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_' + f'nonSBZ_tracked_profiles_{res}_{t_res}_{Np_str}.h5' 
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [96]:
#ColdPool
################################################################

In [99]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def ColdPoolTrackedEDProfile(type,type2):

    if type2=='general':
        profile_array_e=E_G.copy() #PRECALCULATED
        profile_array_d=D_G.copy() #PRECALCULATED
    elif type2=='cloudy':
        profile_array_e=E_C.copy() #PRECALCULATED
        profile_array_d=D_C.copy() #PRECALCULATED

    if type=='all':
        out_arr=ALL_ColdPool_out_arr.copy()
        # after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        # after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        # after_array=DEEP_ColdPool_after_array
    
    
    zhs=data['zh'].values
    e_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    d_profile=e_profile.copy()
    net_profile=e_profile.copy()
    e_profile[:,2]=zhs;d_profile[:,2]=zhs;net_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_e=profile_array_e[ts,p-index_adjust] #JOBARRAY
        np.add.at(e_profile[:, 0], zs, vars_e)
        np.add.at(e_profile[:, 1], zs, 1)

        vars_d=profile_array_d[ts,p-index_adjust] #JOBARRAY
        np.add.at(d_profile[:, 0], zs, vars_d)
        np.add.at(d_profile[:, 1], zs, 1)

        vars_net=vars_e-vars_d
        np.add.at(net_profile[:, 0], zs, vars_net)
        np.add.at(net_profile[:, 1], zs, 1)
    
    print('done\n')
    return e_profile,d_profile,net_profile

In [48]:
for type2 in ['general','cloudy']:
    [ColdPool_ALL_e,ColdPool_ALL_d,ColdPool_ALL_net] = ColdPoolTrackedEDProfile(type='all',type2=type2)
    [ColdPool_SHALLOW_e,ColdPool_SHALLOW_d,ColdPool_SHALLOW_net] = ColdPoolTrackedEDProfile(type='shallow',type2=type2)
    [ColdPool_DEEP_e,ColdPool_DEEP_d,ColdPool_DEEP_net] = ColdPoolTrackedEDProfile(type='deep',type2=type2)
    
    # Define the file path
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f'ColdPool_tracked_profiles_{t_res}_{job_id}.h5'
    
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('ColdPool_ALL_e', data=ColdPool_ALL_e)
        h5f.create_dataset('ColdPool_ALL_d', data=ColdPool_ALL_d)
        h5f.create_dataset('ColdPool_ALL_net', data=ColdPool_ALL_net)
        h5f.create_dataset('ColdPool_SHALLOW_e', data=ColdPool_SHALLOW_e)
        h5f.create_dataset('ColdPool_SHALLOW_d', data=ColdPool_SHALLOW_d)
        h5f.create_dataset('ColdPool_SHALLOW_net', data=ColdPool_SHALLOW_net)
        h5f.create_dataset('ColdPool_DEEP_e', data=ColdPool_DEEP_e)
        h5f.create_dataset('ColdPool_DEEP_d', data=ColdPool_DEEP_d)
        h5f.create_dataset('ColdPool_DEEP_net', data=ColdPool_DEEP_net)


done

done

done



In [49]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [77]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+= f'ColdPool_tracked_profiles_{t_res}_{job_id}.h5'
        
        # Load ColdPool-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir + f'Project_Algorithms/Entrainment/trackout/{type2}_' + f'ColdPool_tracked_profiles.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60
